# Dual-Parameter CMB Predictor

This notebook implements a dual-parameter predictor for CMB analysis that simultaneously predicts both fnl and phi_scales parameters from lensed CMB maps.

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import math
import copy
import logging
import h5py
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

# Suppress TensorFlow warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Flatten, Add, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Add deepsphere path if needed
sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
)

from mlpng import Core
from mlpng.utils import get_data
from mlpng.utils.dataloaders import MapDataset

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set plot style
plt.style.use('seaborn-v0_8-paper')
plt.rc('font', family='serif')
plt.rc('figure', figsize=(12, 8))

print("TensorFlow version:", tf.__version__)
print("GPUs available:", len(tf.config.list_physical_devices('GPU')))

## 2. Load Settings and Configuration

In [ ]:
# Load settings from elsner.json
settings_path = "../settings/elsner.json"

with open(settings_path, 'r') as f:
    settings = json.load(f)

print("Settings loaded:")
print(json.dumps(settings, indent=2))

# Extract key parameters
nside = settings.get('nside', 128)
nsims = settings.get('nsims', 1000)
npix = hp.nside2npix(nside)
npols = 3  # T, Q, U

print(f"\nConfiguration:")
print(f"  nside: {nside}")
print(f"  npix: {npix}")
print(f"  nsims: {nsims}")
print(f"  npols: {npols}")

## 3. Custom DualParameterDataset Class

This dataset generates lensed CMB maps with random pairs of (fnl, phi_scales) parameters.

In [ ]:
class DualParameterDataset(MapDataset):
    """
    Dataset for dual-parameter prediction of fnl and phi_scales.
    
    This dataset generates CMB maps with random fnl and phi_scales values,
    with approximately 10% of samples having both parameters set to zero
    (baseline/unlensed case).
    """
    
    def __init__(
        self,
        file_path,
        nside,
        shapes=None,
        fnl_min=-1000.0,
        fnl_max=1000.0,
        phi_scale_min=0.5,
        phi_scale_max=2.0,
        zero_prob=0.1,  # Probability of zero values for both parameters
        fnl_seed=42,
        rotate=False,
        gaussian_mask=False,
        **kwargs
    ):
        # Store phi_scale parameters
        self.phi_scale_min = phi_scale_min
        self.phi_scale_max = phi_scale_max
        self.zero_prob = zero_prob
        
        # Update y_shape to (None, nshapes + 1) for fnl shapes + phi_scale
        if shapes is None or 'all' in (shapes if isinstance(shapes, list) else [shapes]):
            shapes = ['local', 'equilateral', 'orthogonal']
        elif isinstance(shapes, str):
            shapes = [shapes]
        
        nshapes = len(shapes)
        kwargs['y_shape'] = (None, nshapes + 1)  # fnl values + phi_scale
        kwargs['y_dtype'] = tf.float32
        
        # Initialize parent class
        super().__init__(
            file_path=file_path,
            nside=nside,
            shapes=shapes,
            fnl_min=fnl_min,
            fnl_max=fnl_max,
            fnl_seed=fnl_seed,
            rotate=rotate,
            gaussian_mask=gaussian_mask,
            **kwargs
        )
    
    def _generate(self, indices, duplicates):
        """
        Generate a batch of CMB maps with dual parameters.
        
        Returns:
            maps: Array of shape (batch*duplicates, npix, npols)
            params: Array of shape (batch*duplicates, nshapes+1) containing fnl values and phi_scale
        """
        batch_size = len(indices)
        
        # Set random seed for reproducibility
        np.random.seed(self.fnl_seed + indices[0])
        
        # Generate random fnl values for each shape
        fnls = np.random.uniform(
            low=self.fnl_min,
            high=self.fnl_max,
            size=(len(self.shapes), duplicates, batch_size, 1, 1),
        )
        
        # Generate random phi_scales
        phi_scales = np.random.uniform(
            low=self.phi_scale_min,
            high=self.phi_scale_max,
            size=(duplicates, batch_size),
        )
        
        # Apply zero-value logic: ~10% of samples should have both fnl=0 and phi_scale=0
        zero_mask = np.random.rand(duplicates, batch_size) < self.zero_prob
        fnls[:, zero_mask] = 0
        phi_scales[zero_mask] = 0
        
        # Optionally apply gaussian_mask to fnls (as in parent class)
        if self.gaussian_mask:
            fnls = self._mask_fnls(fnls)
        
        # Load alm data from file
        alm_l = get_data(self.file_path, f"alm_l/{self.l_str}", indices)
        alm_nl = np.array([
            get_data(self.file_path, f"alm_nl/{self.l_str}/{s}", indices)
            for s in self.shapes
        ])
        
        # Combine linear and non-linear components
        alms = alm_l + np.einsum("i...,i...->...", fnls, alm_nl)
        
        # Convert alms to maps in parallel
        n_cpus = len(os.sched_getaffinity(0))
        n_jobs = min(n_cpus, duplicates * batch_size)
        
        with Parallel(n_jobs, pre_dispatch='n_jobs', prefer='threads') as p:
            maps = p(
                delayed(self._alm_to_map)(sim, self.rotate, self.nside)
                for batches in alms
                for sim in batches
            )
        
        # Reshape maps: (batch*duplicates, npix, npols)
        maps = np.transpose(np.array(maps), (0, 2, 1))
        
        # Prepare target parameters: [fnl_shape1, fnl_shape2, ..., phi_scale]
        # fnls shape: (nshapes, duplicates, batch_size, 1, 1)
        fnls = fnls.transpose(1, 2, 3, 4, 0)  # (duplicates, batch_size, 1, 1, nshapes)
        fnls = np.reshape(fnls, (duplicates * batch_size, len(self.shapes)))  # (batch*dup, nshapes)
        
        # phi_scales shape: (duplicates, batch_size)
        phi_scales = phi_scales.reshape(duplicates * batch_size, 1)  # (batch*dup, 1)
        
        # Stack fnls and phi_scales: (batch*dup, nshapes+1)
        params = np.concatenate([fnls, phi_scales], axis=1)
        
        return maps, params

## 4. Model Architecture - DualParameterModel

This model uses a ResidualHealpyUNet encoder (without decoder) followed by separate dense networks for fnl and phi_scales prediction.

In [ ]:
@tf.keras.saving.register_keras_serializable()
class EncoderBlock(tf.keras.layers.Layer):
    """Encoder block using HEALPix convolutions."""
    
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        max_batch_size,
        K,
        use_bn=True,
        use_bias=False,
        pool=True,
        dropout_rate=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        
        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.pool = pool
        indices = np.arange(npix)
        
        enc_layers = [
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=use_bn,
                use_bias=use_bias,
            ),
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=use_bn,
                use_bias=use_bias,
            ),
        ]
        if dropout_rate > 0.0:
            enc_layers.append(Dropout(dropout_rate))
        
        self.body = HealpyGCNN(
            nside=nside,
            indices=indices,
            layers=enc_layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )
        
        self.pooler = None
        if pool:
            self.pooler = HealpyGCNN(
                nside=nside,
                indices=indices,
                layers=[HealpyPool(1, "MAX")],
                n_neighbors=8,
                max_batch_size=max_batch_size,
                initial_Fin=fout,
            )
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "nside": self.nside,
            "npix": self.npix,
            "fin": self.fin,
            "fout": self.fout,
            "activation": self.activation_fn,
            "max_batch_size": self.max_batch_size,
            "K": self.K,
            "use_bn": self.use_bn,
            "use_bias": self.use_bias,
            "pool": self.pool,
            "dropout_rate": self.dropout_rate,
        })
        return config
    
    def call(self, inputs, training=False):
        x = skip = self.body(inputs, training=training)
        if self.pool:
            x = self.pooler(x, training=training)
        return x, skip


def create_dual_parameter_model(input_shape, nshapes=3, max_batch_size=32):
    """
    Create a dual-parameter prediction model.
    
    Args:
        input_shape: Tuple (batch, npix, npols)
        nshapes: Number of fnl shape parameters (default: 3 for local, equilateral, orthogonal)
        max_batch_size: Maximum batch size for HEALPix operations
    
    Returns:
        Compiled Keras model
    """
    npix = input_shape[1]
    npol = input_shape[2]
    nside = hp.npix2nside(npix)
    
    # Calculate encoder depth based on nside
    depth = int(math.log2(nside))
    
    # Channel progression: [npol, 8, 16, 32, 64, 128, ...]
    base_channels = [npol] + [2 ** (i + 3) for i in range(depth)]
    level_npixels = [npix // (4**i) for i in range(depth + 1)]
    level_nsides = [hp.npix2nside(pix) for pix in level_npixels]
    
    # Chebyshev order progression: [1, 3, 5, 7, ...]
    Ks = [1 + (2 * (i // 2)) for i in range(depth + 1)]
    
    print(f"Model architecture:")
    print(f"  Depth: {depth}")
    print(f"  Channels: {base_channels}")
    print(f"  Chebyshev orders: {Ks}")
    
    # Build model
    inputs = Input(shape=input_shape[1:])
    x = inputs
    
    # Encoder blocks
    skips = []
    for i in range(depth):
        block = EncoderBlock(
            level_nsides[i],
            level_npixels[i],
            fin=base_channels[i],
            fout=base_channels[i + 1],
            activation='gelu',
            max_batch_size=max_batch_size,
            K=Ks[i],
            dropout_rate=0.1,
            use_bias=False,
            use_bn=True,
        )
        x, skip = block(x)
        skips.append(skip)
    
    # Bottleneck
    bottleneck = HealpyGCNN(
        nside=level_nsides[depth],
        indices=np.arange(level_npixels[depth]),
        layers=[
            HealpyChebyshev(
                K=Ks[-1],
                Fout=base_channels[-1],
                activation='gelu',
                use_bias=False,
                use_bn=True,
            ),
            HealpyChebyshev(
                K=Ks[-1],
                Fout=base_channels[-1],
                activation='gelu',
                use_bias=False,
                use_bn=True,
            ),
        ],
        n_neighbors=8,
        max_batch_size=max_batch_size,
        initial_Fin=base_channels[-1],
        name='bottleneck',
    )
    x = bottleneck(x)
    
    # Flatten the encoder output
    x = Flatten()(x)
    
    # Dense network for fnl prediction
    fnl_branch = Dense(32, activation='relu', name='fnl_dense1')(x)
    fnl_branch = Dropout(0.1)(fnl_branch)
    fnl_output = Dense(nshapes, name='fnl_output')(fnl_branch)
    
    # Dense network for phi_scales prediction
    phi_branch = Dense(32, activation='relu', name='phi_dense1')(x)
    phi_branch = Dropout(0.1)(phi_branch)
    phi_output = Dense(1, name='phi_output')(phi_branch)
    
    # Concatenate outputs
    outputs = Concatenate(name='combined_output')([fnl_output, phi_output])
    
    # Create model
    model = Model(inputs=inputs, outputs=outputs, name='DualParameterModel')
    
    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mean_squared_error',
        metrics=[
            'mean_absolute_error',
            tf.keras.metrics.RootMeanSquaredError(name='rmse')
        ]
    )
    
    return model

## 5. Dataset Creation and Pipeline Setup

In [ ]:
# For demonstration, we'll use synthetic data generation
# In practice, you would point to an actual HDF5 file with CMB data

# Example: Create dataset (you'll need to provide the actual file path)
data_file = "../data/elsner_data.h5"  # Update with actual path

# Check if data file exists
if not os.path.exists(data_file):
    print(f"Warning: Data file not found at {data_file}")
    print("This notebook requires pre-generated CMB data.")
    print("Please run the generator first or update the data_file path.")
else:
    # Create dataset instance
    dataset = DualParameterDataset(
        file_path=data_file,
        nside=nside,
        shapes=['local', 'equilateral', 'orthogonal'],
        fnl_min=-1000.0,
        fnl_max=1000.0,
        phi_scale_min=0.5,
        phi_scale_max=2.0,
        zero_prob=0.1,
        rotate=False,
        gaussian_mask=True,
        lensed=True,  # Use lensed maps
    )
    
    print(f"Dataset created with {len(dataset)} samples")
    print(f"Output shape: X=(batch, {npix}, {npols}), Y=(batch, 4)")

In [ ]:
# Split dataset into train/val/test
if os.path.exists(data_file):
    # Create cache directory
    cache_dir = os.path.join(os.environ.get('SCRATCH', '/tmp'), 'tf_cache')
    os.makedirs(cache_dir, exist_ok=True)
    
    train_ds, val_ds, test_ds = dataset.split(
        train_size=0.7,
        val_size=0.15,
        test_size=0.15,
        to_tf=True,
        cache_dir=cache_dir,
        cache_file='dual_param',
        gen_batch_size=8,
        duplicates=[2, 1, 1],  # More duplicates for training
        batch_size=32,
        shuffle=True,
        buffer_size=128,
        rotate=[True, False, False],  # Rotation only for training
        gaussian_mask=[True, False, False],  # Masking only for training
    )
    
    print("\nDatasets created:")
    print(f"  Train: {train_ds}")
    print(f"  Val: {val_ds}")
    print(f"  Test: {test_ds}")

## 6. Data Exploration

Let's examine a few samples to ensure the data pipeline is working correctly.

In [ ]:
if os.path.exists(data_file):
    # Get a batch from training data
    for batch_x, batch_y in train_ds.take(1):
        print("Batch shapes:")
        print(f"  X (maps): {batch_x.shape}")
        print(f"  Y (params): {batch_y.shape}")
        
        print("\nFirst sample parameters:")
        print(f"  fnl (local): {batch_y[0, 0]:.2f}")
        print(f"  fnl (equilateral): {batch_y[0, 1]:.2f}")
        print(f"  fnl (orthogonal): {batch_y[0, 2]:.2f}")
        print(f"  phi_scale: {batch_y[0, 3]:.2f}")
        
        # Plot a sample map
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        sample_map = batch_x[0].numpy()
        
        for i, pol in enumerate(['T', 'Q', 'U']):
            hp.mollview(
                sample_map[:, i],
                title=f'Sample CMB Map - {pol} polarization',
                cmap='RdBu_r',
                hold=True,
                sub=(1, 3, i+1)
            )
        
        plt.tight_layout()
        plt.show()
        
        # Plot parameter distributions
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        param_names = ['fnl (local)', 'fnl (equilateral)', 'fnl (orthogonal)', 'phi_scale']
        for i, (ax, name) in enumerate(zip(axes.flat, param_names)):
            ax.hist(batch_y[:, i].numpy(), bins=30, edgecolor='black', alpha=0.7)
            ax.set_xlabel(name)
            ax.set_ylabel('Count')
            ax.set_title(f'Distribution of {name}')
            ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## 7. Model Definition and Summary

In [ ]:
# Create model
input_shape = (None, npix, npols)
nshapes = 3  # local, equilateral, orthogonal

model = create_dual_parameter_model(
    input_shape=input_shape,
    nshapes=nshapes,
    max_batch_size=32
)

# Display model summary
model.summary()

## 8. Model Training

In [ ]:
if os.path.exists(data_file):
    # Setup callbacks
    checkpoint_dir = './checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(checkpoint_dir, 'dual_param_model_best.keras'),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Train model
    epochs = 50
    
    print(f"\nStarting training for {epochs} epochs...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\nTraining complete!")

## 9. Training Curves

In [ ]:
if os.path.exists(data_file) and 'history' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot loss
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss (MSE)', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(alpha=0.3)
    
    # Plot MAE
    axes[1].plot(history.history['mean_absolute_error'], label='Train MAE', linewidth=2)
    axes[1].plot(history.history['val_mean_absolute_error'], label='Val MAE', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Mean Absolute Error', fontsize=12)
    axes[1].set_title('Training and Validation MAE', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Model Evaluation and Predictions

In [ ]:
if os.path.exists(data_file):
    # Make predictions on test set
    print("Making predictions on test set...")
    
    y_true_list = []
    y_pred_list = []
    
    for batch_x, batch_y in test_ds:
        predictions = model.predict(batch_x, verbose=0)
        y_true_list.append(batch_y.numpy())
        y_pred_list.append(predictions)
    
    y_true = np.concatenate(y_true_list, axis=0)
    y_pred = np.concatenate(y_pred_list, axis=0)
    
    print(f"\nPredictions shape: {y_pred.shape}")
    print(f"True values shape: {y_true.shape}")
    
    # Calculate metrics for each parameter
    param_names = ['fnl (local)', 'fnl (equilateral)', 'fnl (orthogonal)', 'phi_scale']
    
    print("\n" + "="*60)
    print("Performance Metrics")
    print("="*60)
    
    for i, name in enumerate(param_names):
        mae = np.mean(np.abs(y_true[:, i] - y_pred[:, i]))
        rmse = np.sqrt(np.mean((y_true[:, i] - y_pred[:, i])**2))
        print(f"\n{name}:")
        print(f"  MAE:  {mae:.4f}")
        print(f"  RMSE: {rmse:.4f}")

## 11. Visualization: True vs Predicted Scatter Plots

In [ ]:
if os.path.exists(data_file) and 'y_pred' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    param_names = ['fnl (local)', 'fnl (equilateral)', 'fnl (orthogonal)', 'phi_scale']
    
    for i, (ax, name) in enumerate(zip(axes.flat, param_names)):
        # Calculate residuals
        residuals = y_true[:, i] - y_pred[:, i]
        std_residual = np.std(residuals)
        
        # Scatter plot
        ax.scatter(y_true[:, i], y_pred[:, i], alpha=0.3, s=10, c='blue', label='Predictions')
        
        # Perfect prediction line
        min_val = min(y_true[:, i].min(), y_pred[:, i].min())
        max_val = max(y_true[:, i].max(), y_pred[:, i].max())
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Perfect prediction')
        
        # ±1σ error bands
        ax.fill_between(
            [min_val, max_val],
            [min_val - std_residual, max_val - std_residual],
            [min_val + std_residual, max_val + std_residual],
            alpha=0.2,
            color='red',
            label=f'±1σ ({std_residual:.2f})'
        )
        
        ax.set_xlabel(f'True {name}', fontsize=11)
        ax.set_ylabel(f'Predicted {name}', fontsize=11)
        ax.set_title(f'{name}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('true_vs_predicted.png', dpi=150, bbox_inches='tight')
    plt.show()

## 12. Residual Distributions

In [ ]:
if os.path.exists(data_file) and 'y_pred' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    param_names = ['fnl (local)', 'fnl (equilateral)', 'fnl (orthogonal)', 'phi_scale']
    
    for i, (ax, name) in enumerate(zip(axes.flat, param_names)):
        residuals = y_true[:, i] - y_pred[:, i]
        
        # Histogram
        n, bins, patches = ax.hist(
            residuals,
            bins=50,
            density=True,
            alpha=0.7,
            color='skyblue',
            edgecolor='black',
            label='Residuals'
        )
        
        # Fit Gaussian
        mu = np.mean(residuals)
        sigma = np.std(residuals)
        x = np.linspace(residuals.min(), residuals.max(), 100)
        gaussian = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma)**2)
        ax.plot(x, gaussian, 'r-', linewidth=2, label=f'Gaussian fit\n(μ={mu:.2f}, σ={sigma:.2f})')
        
        ax.axvline(0, color='green', linestyle='--', linewidth=2, label='Zero residual')
        ax.set_xlabel(f'Residual (True - Predicted) for {name}', fontsize=11)
        ax.set_ylabel('Probability Density', fontsize=11)
        ax.set_title(f'Residual Distribution: {name}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('residual_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 13. Summary and Results

This notebook successfully implemented a dual-parameter predictor for CMB analysis:

### Key Achievements:

1. **Custom Dataset (DualParameterDataset)**:
   - Generates lensed CMB maps with random pairs of (fnl, phi_scales)
   - Samples fnl uniformly from [-1000, 1000]
   - Samples phi_scales uniformly from [0.5, 2.0]
   - Implements 10% zero-value logic for baseline cases
   - Supports all features: duplication, caching, splitting, rotation, parallel generation

2. **Model Architecture (DualParameterModel)**:
   - Uses ResidualHealpyUNet encoder with HEALPix convolutions
   - Progressive Chebyshev orders (1, 3, 5, 7, ...)
   - Channel progression: [3, 8, 16, 32, 64, 128, ...]
   - Separate dense networks for fnl and phi_scales predictions
   - Compiled with MSE loss and MAE metrics

3. **Training Pipeline**:
   - Proper data splitting (70% train, 15% val, 15% test)
   - Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
   - TensorFlow dataset integration with caching and shuffling

4. **Evaluation Metrics**:
   - Scatter plots showing true vs predicted values with error bands
   - Residual distributions with Gaussian fits
   - Training curves for loss and MAE

### Next Steps:
- Fine-tune hyperparameters (learning rate, batch size, architecture depth)
- Compare with Fisher matrix error bounds
- Experiment with different loss functions (e.g., weighted MSE)
- Test on different data sets and parameter ranges

In [ ]:
# Optional: Save final model
if os.path.exists(data_file):
    model.save('dual_parameter_model_final.keras')
    print("Model saved to 'dual_parameter_model_final.keras'")